In [3]:
projname = !gcloud config get-value project

In [10]:
PROJECT_ID = projname[0]
REGION = "us-central1"
BUCKET = "shaydpbucket2"

In [7]:
!gcloud config set project {PROJECT_ID}

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [8]:
!gcloud services enable dataproc.googleapis.com storage.googleapis.com

Operation "operations/acat.p2-1096302915807-1ce617a4-9aa2-44aa-9ec8-45525ef35f11" finished successfully.


In [11]:
!gsutil mb -l {REGION} gs://{BUCKET}

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Creating gs://shaydpbucket2/...


In [12]:
%%writefile wordcount.py
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('wordcount-demo').getOrCreate()
data = spark.createDataFrame([
    ("hello spark",),
    ("hello Elad",),
    ("hello spark",),
    ("hello spark dataproc serveless",),],["text"])

words = (data.rdd.flatMap(lambda row:row.text.split())
.map(lambda word:(word,1))
.reduceByKey(lambda a,b:a+b)
)

result = words.toDF(["word","count"])
result.show()
spark.stop()


Writing wordcount.py


In [13]:
!gsutil cp wordcount.py gs://{BUCKET}/code/wordcount.py

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://wordcount.py [Content-Type=text/x-python]...
/ [1 files][  434.0 B/  434.0 B]                                                
Operation completed over 1 objects/434.0 B.                                      


In [14]:
!gcloud dataproc batches submit pyspark gs://{BUCKET}/code/wordcount.py \
--region={REGION} \
--properties=spark.dynamicAllocation.enabled=true \
--properties=spark.dynamicAllocation.initialExecutors=3 \
--properties=spark.dynamicAllocation.minExecutors=3 \
--properties=spark.dynamicAllocation.maxExecutors=10


Batch [33a13fe7eed149a493b79b4bf9c2caf6] submitted.
Using the default container image
Waiting for container log creation
PYSPARK_PYTHON=/opt/dataproc/conda/bin/python
Generating /home/spark/.pip/pip.conf
Configuring index-url as 'https://us-python.pkg.dev/artifact-registry-python-cache/virtual-python/simple/'
JAVA_HOME=/usr/lib/jvm/temurin-17-jdk-amd64
SPARK_EXTRA_CLASSPATH=
:: loading settings :: file = /etc/spark/conf/ivysettings.xml
26/06/16 15:48:17 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-google-hadoop-file-system.properties,hadoop-metrics2.properties
26/06/16 15:48:17 INFO MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
26/06/16 15:48:17 INFO MetricsSystemImpl: google-hadoop-file-system metrics system started
+---------+-----+
|     word|count|
+---------+-----+
|    hello|    4|
|     Elad|    1|
|serveless|    1|
|    spark|    3|
| dataproc|    1|
+---------+-----+

Batch [33a13fe7eed149a493b79b4bf9c2caf6] finished.
metadata:
